# Web Search & Content Extraction Tools — Demo

Quick demos of `google_search` and `fetch_page_as_markdown`.

In [4]:
import sys, json
sys.path.insert(0, '..')

from tools.web_search import GoogleSearchInput, GoogleSearchResult, google_search
from tools.web_search_ddg import DuckDuckGoSearchInput, DuckDuckGoSearchResult, duckduckgo_search
from tools.web_content import FetchPageInput, PageMarkdownResult, fetch_page_as_markdown
from IPython.display import Markdown, display

## 1. Fetch example.com — Simplest possible test

In [ ]:
args = FetchPageInput(url="https://example.com")
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown)} chars")
print("---")
display(Markdown(result.markdown))

## 2. Google Search — "Pikachu pokemon"

In [ ]:
args = GoogleSearchInput(query="Pikachu pokemon", max_results=5)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Error: {result.error}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:120]}")
    print()

## 3. Google Search — Site-restricted to Bulbapedia

In [ ]:
args = GoogleSearchInput(
    query="Charizard",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=5,
)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}\n")

## 4. Fetch Bulbapedia — Pikachu page (stealth mode)

Bulbapedia is behind Cloudflare, so we use `use_stealth=True`.

In [ ]:
args = FetchPageInput(
    url="https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True,
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
# Show first 2000 chars as rendered markdown
display(Markdown(result.markdown[:20000] + "\n\n*... (truncated) ...*"))

## 5. Fetch Wikipedia — Clean content extraction

In [ ]:
args = FetchPageInput(
    url="https://en.wikipedia.org/wiki/Pok%C3%A9mon",
    css_selector="#bodyContent",
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
display(Markdown(result.markdown[:3000] + "\n\n*... (truncated) ...*"))

## 6. Search → Fetch pipeline

Search for something, then fetch the first result and show its markdown.

In [ ]:
# Step 1: Search
search_args = GoogleSearchInput(
    query="Bulbasaur",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=1,
)
search_raw = google_search(search_args)
search_result = GoogleSearchResult.model_validate_json(search_raw)

if search_result.results:
    first = search_result.results[0]
    print(f"Top result: {first.title}")
    print(f"URL: {first.url}\n")

    # Step 2: Fetch that page
    fetch_args = FetchPageInput(
        url=first.url,
        css_selector="#mw-content-text",
        use_stealth=True,
    )
    fetch_raw = fetch_page_as_markdown(fetch_args)
    page = PageMarkdownResult.model_validate_json(fetch_raw)

    print(f"Page title: {page.title}")
    print(f"Content length: {len(page.markdown):,} chars")
    print("---")
    display(Markdown(page.markdown[:5000] + "\n\n*... (truncated) ...*"))
else:
    print("No results found.")

## 9. DuckDuckGo Search — "Bulbasaur site:bulbapedia.bulbagarden.net"

DuckDuckGo is used here as an alternative to Google Search, often providing better snippets for automated retrieval.

In [7]:
args = DuckDuckGoSearchInput(
    query="Bulbasaur",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=5
)
raw = duckduckgo_search(args)
result = DuckDuckGoSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:1000]}...")
    print()

[2026-03-20 22:20:09] INFO: Fetched (200) <GET https://duckduckgo.com/?q=site%3Abulbapedia.bulbagarden.net+Bulbasaur&t=h_&ia=web> (referer: https://www.google.com/)


Query: site:bulbapedia.bulbagarden.net Bulbasaur
Results: 5

[1] Bulbasaur (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
    https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)
    3 days ago Bulbasaur (Japanese: フシギダネ Fushigidane) is a dual-type Grass / Poison Pokémon introduced in Generation I. It evolves into Ivysaur starting at level 16, which evolves into Venusaur starting at level 32. Along with Charmander and Squirtle, Bulbasaur is one of the three first partner Pokémon of Kanto available at the beginning of Pokémon Red, Green, Blue, FireRed, and LeafGreen....

[2] Bulbasaur (Pokémon)/Generation I learnset - Bulbapedia, the community ...
    https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)/Generation_I_learnset
    Nov 6, 2024 By transfer from another generation Category: Pokémon learnsets (Generation I)...

[3] Bulbasaur (Pokémon)/Generation III learnset - Bulbapedia, the community ...
    https://bulbapedia.bulbagarden.net/wiki/

## 7. Vector Database Ingestion

Ingesting a webpage into the vector database using Chonkie for chunking and ChromaDB for storage.

In [ ]:
from tools.web_vector_db import ingest_web_page, IngestWebPageArgs

args = IngestWebPageArgs(
    url="https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True
)
result = ingest_web_page(args)
print(result)

## 8. Vector Database Querying

Retrieving semantically relevant chunks from the ingested web corpus.

In [ ]:
from tools.web_vector_db import query_web_content, QueryWebContentArgs

query_args = QueryWebContentArgs(
    query="What is the flame on Charmander's tail?",
    n_results=10
)
query_result = query_web_content(query_args)
display(Markdown(query_result))